# Scene Lab — VISUAL ONLY

No beats, no Acts, no narration timing. One question: **can the model make a good animated visual?**
Edit the brief + prompt, run down the cells, watch. Everything animates off the frame clock.


In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

d = os.getcwd()
while d != os.path.dirname(d):
    if os.path.isdir(os.path.join(d, 'apps', 'backend', 'decode')):
        ROOT = Path(d); break
    d = os.path.dirname(d)
sys.path.insert(0, str(ROOT / 'apps' / 'backend'))

from decode.config import get_settings
from dotenv import load_dotenv
load_dotenv(ROOT / 'apps' / 'backend' / '.env')   # notebook runs from notebooks/
from openai import OpenAI

S = get_settings()
client = OpenAI(api_key=S.openai_api_key, base_url=S.openai_base_url or None)
print('model:', S.openai_model)


## 1. The brief — EDIT THIS

`TOPIC` = what it teaches. `SHOW` = the concrete picture you want. That's the whole input.


In [ ]:
TOPIC = "How a bloom filter tests set membership"
SHOW = ("A horizontal row of bit cells (0/1). Three hash functions map an item to three cells and set "
        "them to 1. Then a lookup checks an item's three cells: if any is 0 -> definitely not in the set; "
        "if all are 1 -> probably in the set.")
PALETTE = {"surface":"#f5f3ee","border":"#d8d3c8","ink":"#1b1b1b","support":"#6c6c6c","accent":"#2f6bff"}
DUR = 12.0  # seconds


## 2. The prompt — EDIT THIS

The entire instruction. Pure visual: the picture, the tools, the rules. No timing machinery.


In [ ]:
SYSTEM = "You are a motion designer. You build ONE beautiful animated Remotion scene as a React component. You reach for the right library instead of hand-drawing pixels."

PROMPT = f"""Build ONE 1920x1080 animated scene that clearly teaches: {TOPIC}

The picture to build:
{SHOW}

Palette: {json.dumps(PALETTE)}

TOOLS — import from `@decode/animation-api` (it re-exports all of these):
  d3, THREE, ThreeCanvas, gsap (via useGsapTimeline), Lottie,
  paths (paths.evolvePath(progress, d) to draw a path on), shapes (shapes.Rect/Circle/...),
  roughNotation (Circle/Underline/Highlight/Box), noise, effects,
  and Remotion: AbsoluteFill, useCurrentFrame, useVideoConfig, interpolate, spring, Easing.

RULES:
- Default-export `function Scene()`. Take NO props.
- Animate EVERYTHING off useCurrentFrame() / useVideoConfig() spread across the full duration
  (durationInFrames). This is a VIDEO not a slide: the main subject MOVES continuously across the whole duration (a position travelling, a value tweening, a shape morphing, a path drawing on) — not things that pop in and hold still. Elements enter with motion. No timers, no CSS
  animation, no Math.random.
- Pick the ONE clear picture; show the mechanism actually working, large and centre-stage.
- LIBRARY FIRST: never hand-place elements with absolute left/top pixels; never hand-write SVG path
  strings (d="M .. L .."). A curve is d3 + paths.evolvePath; a shape is shapes; a mark is roughNotation.
- Every <svg> has a viewBox. Text is HTML (<div>), never <svg><text>. Paint your own background from
  the palette.
- Return ONLY the .tsx code in one ```tsx block. No prose."""

print(PROMPT)


## 3. The vision loop — generate → render still → critique → patch → repeat

The model never ships a frame it hasn't seen. Each iteration renders 3 stills (early/mid/late),
sends them BACK to the model, gets spatial defects, and patches. Caps at `MAX_ITERS`.


In [ ]:
import base64, shutil
from pydantic import BaseModel
fe = ROOT / "apps" / "frontend"
MAX_ITERS = 4

def ask(system, user):
    r = client.chat.completions.create(model=S.openai_model,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    raw = r.choices[0].message.content
    m = re.search(r"```(?:tsx|jsx|ts|js)?\n(.*?)```", raw, re.DOTALL)
    return (m.group(1) if m else raw).strip()

def render_stills(source):
    props={"scenes":[{"id":"lab","title":TOPIC,"dur":DUR,"componentSource":source,"controls":[],"words":[]}],
           "visualPick":{},"palette":PALETTE}
    (fe/"scenes.json").write_text(json.dumps(props))
    work=fe/".data"/"vision"/"lab"; shutil.rmtree(work,ignore_errors=True); work.mkdir(parents=True,exist_ok=True)
    r=subprocess.run(["npx","tsx","scripts/render-stills.ts","scenes.json",str(work)],
                     cwd=fe,capture_output=True,text=True)
    pngs=sorted(work.glob("p*.png"))
    if not pngs: raise RuntimeError("still render failed:\n"+r.stderr[-800:])
    # Scenes that crash render the error panel (not a throw); the still script
    # prints the real reason as "SCENE_ERROR ...". Collect them so the loop can
    # feed the exception back and fix the CODE, not just the layout.
    errors=[ln[len("SCENE_ERROR "):] for ln in r.stdout.splitlines() if ln.startswith("SCENE_ERROR ")]
    return pngs, errors

class Critique(BaseModel):
    ok: bool
    defects: list[str]

def critique(pngs):
    imgs=[{"type":"input_image","image_url":"data:image/png;base64,"+base64.b64encode(p.read_bytes()).decode()} for p in pngs]
    resp=client.responses.parse(model=S.openai_model,
      instructions=("You review THREE frames (early, middle, late) of an educational animation as a viewer sees them. "
        "Report SPATIAL defects only: elements overlapping, anything clipped at the frame edge, big dead space, "
        "content huddled small instead of filling the stage, unreadable/tiny text, an empty or near-empty frame. "
        "Minor imperfection is fine. Set ok=true only if a viewer would not notice a problem. "
        "Each defect: what and where, concretely, so an animator can fix it."),
      input=[{"role":"user","content":[{"type":"input_text","text":f"Intended visual: {TOPIC}. {SHOW}"},*imgs]}],
      text_format=Critique)
    return resp.output_parsed



## 4. Run the loop


In [ ]:
source = ask(SYSTEM, PROMPT)
print(f"gen: {len(source)} chars")
for i in range(MAX_ITERS):
    pngs, errors = render_stills(source)
    if errors:
        print(f"iter {i}: RUNTIME ERROR -> {errors}")
        fix = (f"Current scene code:\n```tsx\n{source}\n```\n\n"
               "It threw this runtime error when rendered:\n" + "\n".join(errors) +
               "\n\nFix the bug. Keep the visual idea. Return the complete corrected .tsx in one ```tsx block.")
        source = ask(SYSTEM, fix)
        print(f"  patched -> {len(source)} chars")
        continue
    c = critique(pngs)
    print(f"iter {i}: ok={c.ok}" + ("" if c.ok else "  defects:\n   - " + "\n   - ".join(c.defects)))
    if c.ok: break
    patch = (f"Current scene code:\n```tsx\n{source}\n```\n\n"
             "A design reviewer looked at the rendered frames and found these SPATIAL defects:\n- "
             + "\n- ".join(c.defects) +
             "\n\nFix ONLY these. Keep the visual idea, libraries, and structure. "
             "Return the complete corrected .tsx in one ```tsx block.")
    source = ask(SYSTEM, patch)
    print(f"  patched -> {len(source)} chars")


## 5. Render the final video


In [ ]:
props={"scenes":[{"id":"lab","title":TOPIC,"dur":DUR,"componentSource":source,"controls":[],"words":[]}],"visualPick":{},"palette":PALETTE}
(fe/"scenes.json").write_text(json.dumps(props,indent=2))
if (fe/"lab.mp4").exists(): (fe/"lab.mp4").unlink()
r=subprocess.run(["npx","tsx","scripts/render-video.ts","scenes.json","lab.mp4"],cwd=fe,capture_output=True,text=True)
print("OK" if (fe/"lab.mp4").exists() else "FAILED\n"+r.stderr[-1200:])
from IPython.display import Video
Video(str(fe/"lab.mp4"), embed=True, width=960)


---
## Reference: the real pipeline prompts

Not used by the lab above — this is what `renderer.generate` sends for the full video. Kept here
to inspect for contradictions.


In [ ]:
from decode.agents.renderer import build as build_renderer
from decode.agents.renderer import prompt as rprompt
r = build_renderer(S)
print("=== SYSTEM ===\n", r.runtime.system())
print("\n\n=== GUIDANCE (build_instructions user message) ===\n", rprompt.CHOREOGRAPHY_GUIDANCE)
